# GenAI Lab: LangChain + Cohere Document Q&A / Summarization

This notebook demonstrates:
1. Installing `langchain`, `cohere`, and `langchain-community`
2. Loading a text document from Google Drive
3. Using a Cohere LLM with a custom `PromptTemplate` to format output in a structured way

## Step 1: Install required packages

In [ ]:
!pip install langchain cohere langchain-community -q

## Step 2: Mount Google Drive

Run this in Google Colab. It will ask you to authorize access to your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 3: Set your Cohere API key

Get a free key from https://dashboard.cohere.com/api-keys and paste it below.
(For a real demo, consider using `getpass` instead of hardcoding it.)

In [ ]:
import os
from getpass import getpass

os.environ["COHERE_API_KEY"] = getpass("Enter your Cohere API key: ")

## Step 4: Load a text document from Google Drive

Update `file_path` to point to the `.txt` file inside your Drive.

In [ ]:
from langchain_community.document_loaders import TextLoader

# Update this path to match where your file lives in Drive
file_path = "/content/drive/MyDrive/genai_lab/sample.txt"

loader = TextLoader(file_path)
documents = loader.load()

print(f"Loaded {len(documents)} document(s).\n")
print("Preview:\n", documents[0].page_content[:500])

## Step 5: Create a Prompt Template

This defines exactly how the model's output should be structured — useful for a clean lab demo.

In [ ]:
from langchain.prompts import PromptTemplate

prompt_template = PromptTemplate(
    input_variables=["document_text"],
    template="""
You are an intelligent assistant. Read the following text and respond ONLY in this exact format:

TITLE: <a short, descriptive title for the text>
SUMMARY: <a 3-4 line summary of the text>
KEY POINTS:
- <key point 1>
- <key point 2>
- <key point 3>
SENTIMENT: <Positive / Negative / Neutral>

Text:
{document_text}
"""
)

## Step 6: Run the chain with Cohere

In [ ]:
from langchain_community.chat_models import ChatCohere

llm = ChatCohere(model="command-r", temperature=0.3)

chain = prompt_template | llm

response = chain.invoke({"document_text": documents[0].page_content})

print(response.content)

---
### Notes for the demo
- If `ChatCohere` isn't available in your installed version, swap in `from langchain_community.llms import Cohere` and use `llm = Cohere(model="command")` instead — the rest of the chain stays the same.
- You can point `file_path` at any `.txt` file you upload to your own Drive under a folder like `genai_lab/`.
- Try changing the template in Step 5 (e.g. add a `TRANSLATION:` field) to show the class how easy it is to control output structure.